In [70]:
import  pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import (train_test_split,StratifiedKFold,GridSearchCV,RandomizedSearchCV,cross_val_score)
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report,confusion_matrix,accuracy_score
from sklearn.tree import DecisionTreeClassifier
from scipy.stats import randint

In [71]:
df = pd.read_csv('bank_marketing.csv')
df.head(5)

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,32,admin.,married,secondary,no,11434,no,no,cellular,14,jan,152,3,259,10,unknown,no
1,87,student,single,secondary,no,-1149,no,yes,cellular,9,mar,911,11,195,2,failure,no
2,62,housemaid,single,primary,no,14361,no,no,cellular,27,nov,2563,12,-1,0,unknown,yes
3,23,unemployed,single,secondary,no,28512,no,no,cellular,27,jun,696,12,-1,0,unknown,yes
4,27,housemaid,married,secondary,no,58589,yes,no,cellular,18,apr,2834,11,-1,0,unknown,yes


In [72]:
df.shape

(20000, 17)

In [73]:
df.isnull().sum()

age          0
job          0
marital      0
education    0
default      0
balance      0
housing      0
loan         0
contact      0
day          0
month        0
duration     0
campaign     0
pdays        0
previous     0
poutcome     0
y            0
dtype: int64

In [74]:
df.dtypes

age          int64
job            str
marital        str
education      str
default        str
balance      int64
housing        str
loan           str
contact        str
day          int64
month          str
duration     int64
campaign     int64
pdays        int64
previous     int64
poutcome       str
y              str
dtype: object

In [75]:
df.head()

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,32,admin.,married,secondary,no,11434,no,no,cellular,14,jan,152,3,259,10,unknown,no
1,87,student,single,secondary,no,-1149,no,yes,cellular,9,mar,911,11,195,2,failure,no
2,62,housemaid,single,primary,no,14361,no,no,cellular,27,nov,2563,12,-1,0,unknown,yes
3,23,unemployed,single,secondary,no,28512,no,no,cellular,27,jun,696,12,-1,0,unknown,yes
4,27,housemaid,married,secondary,no,58589,yes,no,cellular,18,apr,2834,11,-1,0,unknown,yes


In [76]:
df.columns = df.columns.str.strip()

In [77]:
df.columns

Index(['age', 'job', 'marital', 'education', 'default', 'balance', 'housing',
       'loan', 'contact', 'day', 'month', 'duration', 'campaign', 'pdays',
       'previous', 'poutcome', 'y'],
      dtype='str')

In [78]:
# Target is "y"
traget = 'y'

print(df[traget].value_counts())

y
yes    13307
no      6693
Name: count, dtype: int64


In [79]:
# Label Encoding for multiple categorical columns

for col in df.select_dtypes(include='str').columns:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])


In [80]:
df.head(5)

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,32,0,1,1,0,11434,0,0,0,14,4,152,3,259,10,3,0
1,87,8,2,1,0,-1149,0,1,0,9,7,911,11,195,2,0,0
2,62,3,2,0,0,14361,0,0,0,27,9,2563,12,-1,0,3,1
3,23,10,2,1,0,28512,0,0,0,27,6,696,12,-1,0,3,1
4,27,3,1,1,0,58589,1,0,0,18,0,2834,11,-1,0,3,1


In [81]:
# Features and Target
x = df.drop(columns=[traget])
y = df[traget]
print(x.shape, y.shape)

(20000, 16) (20000,)


In [82]:
# Train Test Split
# Train = 70% and Test = 30%
x_train, x_test, y_train, y_test = train_test_split(x,y,test_size=0.3,random_state=42)

print("x train shape :", x_train.shape)
print("x test shape :", x_test.shape)
print("y train shape :", y_train.shape)
print("y test shape :", y_test.shape)

x train shape : (14000, 16)
x test shape : (6000, 16)
y train shape : (14000,)
y test shape : (6000,)


In [83]:
decision_tree = DecisionTreeClassifier(random_state=42)
decision_tree.fit(x_train, y_train)

print("Accuracy:", accuracy_score(y_test, decision_tree.predict(x_test)))
print("Classification Report:")
print(classification_report(y_test, decision_tree.predict(x_test)))
print("Confusion Matrix:")
print(confusion_matrix(y_test, decision_tree.predict(x_test)))

Accuracy: 0.7281666666666666
Classification Report:
              precision    recall  f1-score   support

           0       0.58      0.65      0.61      1977
           1       0.82      0.77      0.79      4023

    accuracy                           0.73      6000
   macro avg       0.70      0.71      0.70      6000
weighted avg       0.74      0.73      0.73      6000

Confusion Matrix:
[[1278  699]
 [ 932 3091]]


In [84]:
# Cross Validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Perform 5-fold cross-validation
cv_scores = cross_val_score(decision_tree, x, y,cv=cv, scoring='accuracy')

print("Cross-Validation Scores:", cv_scores)
print("Mean CV Score:", round(cv_scores.mean(), 4))
print("Standard Deviation of CV Scores:", round(cv_scores.std(), 4))

Cross-Validation Scores: [0.73175 0.72075 0.7305  0.7225  0.729  ]
Mean CV Score: 0.7269
Standard Deviation of CV Scores: 0.0044


In [92]:
grid_search_params = {
    'criterion': ['gini', 'entropy'],
    'max_depth': [2, 5, 10, 15, 20, None], 
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4]
}

grid_search = GridSearchCV(estimator=decision_tree, param_grid=grid_search_params, cv=5, scoring='accuracy', n_jobs=-1)
grid_search.fit(x_train, y_train)
print("Best Parameters:", grid_search.best_params_)
print("Best Cross-Validation Score:", grid_search.best_score_)
print("Test Set Accuracy with Best Parameters:", accuracy_score(y_test, grid_search.predict(x_test)))

Best Parameters: {'criterion': 'entropy', 'max_depth': 10, 'min_samples_leaf': 1, 'min_samples_split': 5}
Best Cross-Validation Score: 0.8185
Test Set Accuracy with Best Parameters: 0.8356666666666667


In [95]:
random_search = RandomizedSearchCV(estimator=decision_tree, param_distributions={
    'max_depth': [3, 5, 10, None],
    'min_samples_split': randint(2, 20),
    'min_samples_leaf': randint(1, 20),
    'criterion': ['gini', 'entropy']
}, n_iter=100, cv=5, scoring='accuracy', random_state=42,n_jobs=-1)

random_search.fit(x_train, y_train)

print("Best Parameters from Randomized Search:", random_search.best_params_)
print("Best Cross-Validation Score from Randomized Search:", random_search.best_score_)
print("Test Set Accuracy with Best Parameters from Randomized Search:", accuracy_score(y_test, random_search.predict(x_test)))

Best Parameters from Randomized Search: {'criterion': 'entropy', 'max_depth': 10, 'min_samples_leaf': 3, 'min_samples_split': 2}
Best Cross-Validation Score from Randomized Search: 0.8185714285714287
Test Set Accuracy with Best Parameters from Randomized Search: 0.8355
